## Modeling Plan
### Target:
- `Churn`

### Excluded columns:
- `customerID`

### Numeric features:
- `tenure`
- `MonthlyCharges`
- `TotalCharges`

### Categorical features:
- all remaining non-target, non-ID columns

### Deterministic cleaning:
- convert `TotalCharges` to numeric
- set blank `TotalCharges` values to 0 based on the tenure=0 finding

### Preprocessing:
- numeric imputation/scaling
- categorical imputation/one-hot encoding

### Baseline models:
- DummyClassifier
- LogisticRegression

### Primary evaluation focus:
- compare against dummy baseline
- use accuracy together with precision, recall, F1, ROC-AUC, and PR-AUC

In [111]:
## Imports
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [112]:
## Load Data
data_dir = Path.cwd().parent / 'data'
raw_data = (data_dir / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(raw_data)
df.head(n=10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [113]:
## Deterministic Cleaning
clean_df = df.copy()

# Filling all missing 'TotalCharges' with 0's as their 'tenure' = 0
clean_df['TotalCharges']  = pd.to_numeric(clean_df['TotalCharges'], errors='coerce')
clean_df.fillna(0, inplace=True)
print(clean_df['TotalCharges'].eq(' ').sum())

0


In [114]:
## Define Features and Target
y = clean_df['Churn'].map({'No':0, 'Yes':1})
X = clean_df.drop(columns=["customerID", "Churn"])

In [115]:
## Train/Test Split
test_size=0.2
random_state=42

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=test_size,
                                                    random_state=random_state,
                                                    stratify=y
                                                   )

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(y_train.value_counts(normalize=True))

(5634, 19) (1409, 19) (5634,) (1409,)
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64


In [116]:
## Preprocessing Plan

numeric_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

categorical_cols = [
    cols for cols in X.columns if cols not in numeric_cols
]

assert(len(numeric_cols)+len(categorical_cols) == X_train.shape[1]), "numeric_cols + categorical_cols = all cols"

numeric_preprocessor = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy="median")),
        ('scaler', StandardScaler())
         ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy="most_frequent")),
         ('onehot', OneHotEncoder(handle_unknown='ignore'))
          ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocessor, numeric_cols),
        ('cat', categorical_preprocessor, categorical_cols)
    ]
)

In [117]:
## Aux Functions
def eval_model(model, X, y, model_name='model'):
    y_true = y
    y_model = model.predict(X)
    y_model_proba = model.predict_proba(X_test)[:,-1]
    
    metrics = {
        'model_name' : model_name,
        'accuracy_score' : accuracy_score(y_true, y_model),
        'recall_score' : recall_score(y_true, y_model),
        'precision_score' : precision_score(y_true, y_model, zero_division=0),
        'f1_score' : f1_score(y_true, y_model, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_model_proba),
        'pr_auc': average_precision_score(y_true, y_model_proba)
    }
    
    metrics_df = pd.DataFrame([metrics])
    
    print(f"Model: {model_name}")
    print(f"\nmetrics :")
    display(metrics_df)

    cf_mat = confusion_matrix(y_true, y_model)
    cf_df = pd.DataFrame(
        data=cf_mat,
        index=['True not Churn', 'True Churn'],
        columns=['Predicted not Churn', 'Predicted Churn']
    )

    print(f"\nConfusion Metrix :")
    display(cf_df)

    print(f"\n Classification Report:")
    print(classification_report(y_true, y_model, zero_division=0))

    return metrics_df, cf_df



In [122]:
## Dummy Baseline
# The dummy baseline predicts the majority class from the training data (not churned).

dummy_pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('model', DummyClassifier(strategy="most_frequent"))
    ]
)

dummy_pipeline.fit(X_train, y_train)

dummy_metrics, dummy_cf_mat = eval_model(dummy_pipeline, X_test, y_test, "dummy")

Model: dummy

metrics :


,model_name,accuracy_score,recall_score,precision_score,f1_score,roc_auc,pr_auc
0,dummy,0.734564,0.0,0.0,0.0,0.5,0.265436



Confusion Metrix :


,Predicted not Churn,Predicted Churn
True not Churn,1035,0
True Churn,374,0



 Classification Report:
              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1035
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409



## Dummy Baseline Results

The dummy model predicts the majority class, `not churn`, for every customer.

This gives an accuracy of about 73.5%, which matches the majority-class proportion in the test set. However, the model identifies no churn customers:

- churn recall: 0.0
- churn precision: 0.0
- churn F1-score: 0.0

The confusion matrix shows that all 374 churn customers in the test set are incorrectly predicted as not churn.
ROC_AUC & PR_AUC give the expected values for dummy basline : 50% & 26% (Coin flipping guessing and Churn proportion in the dataset respectively) 

In [119]:
## Logistic Regression Baseline
log_reg_pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('model', 
         LogisticRegression(
            max_iter=1000,
            random_state=random_state)
        )
    ]
)

log_reg_pipeline.fit(X_train,y_train)

log_reg_metrics, log_reg_cf_mat = eval_model(log_reg_pipeline, X_test, y_test, "logistic regression")

Model: logistic regression

metrics :


,model_name,accuracy_score,recall_score,precision_score,f1_score,roc_auc,pr_auc
0,logistic regression,0.805536,0.558824,0.657233,0.604046,0.842034,0.633702



Confusion Metrix :


,Predicted not Churn,Predicted Churn
True not Churn,926,109
True Churn,165,209



 Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



## Logistic Regression Baseline Results

The logistic regression baseline significantly improves over the dummy classifier.

Compared with the dummy baseline, logistic regression improves:

- accuracy from about 73.5% to about 80.6%
- churn recall from 0.0 to about 55.9%
- churn precision from 0.0 to about 65.7%
- churn F1-score from 0.0 to about 60.4%
- ROC-AUC from 0.50 to about 0.84
- PR-AUC from about 0.27 to about 0.63

The confusion matrix shows that the model correctly identifies 209 out of 374 churn customers, but still misses 165 churn customers.

This means the baseline model has learned meaningful churn patterns, but recall may still need improvement depending on the business goal.

In [120]:
# PrecisionRecallDisplay.from_estimator(log_reg_pipeline, X_test, y_test, drop_intermediate=True)

In [121]:
## Baseline Model Comparison
baseline_results = pd.concat([dummy_metrics, log_reg_metrics], ignore_index=True) \
    .set_index('model_name') \
    .round(3)

baseline_results

,accuracy_score,recall_score,precision_score,f1_score,roc_auc,pr_auc
model_name,,,,,,
dummy,0.735,0.000,0.000,0.000,0.500,0.265
logistic regression,0.806,0.559,0.657,0.604,0.842,0.634


## Baseline Model Comparison
The logistic regression baseline substantially improves over the dummy classifier.

The dummy model achieves about 73.5% accuracy by always predicting the majority class, but it identifies no churn customers. Logistic regression improves accuracy to about 80.6%, identifies about 55.9% of churn customers, and improves PR-AUC from about 0.27 to about 0.63.

This shows that the model learned meaningful churn patterns, but the default threshold still misses many churners. Future steps should evaluate model stability using cross-validation and then analyze threshold tradeoffs.

In [164]:
## Kfold CV
random_state=42
kfold=5

cv = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=random_state)

scoring = {
        'accuracy': "accuracy",
        'recall' : "recall",
        'precision' : "precision",
        'f1' : "f1",
        'roc_auc': "roc_auc",
        'pr_auc': "average_precision"
    }

metrics = cross_validate(log_reg_pipeline, 
                         X_train, 
                         y_train, 
                         cv=cv, 
                         scoring=scoring, 
                         return_train_score=False
                        )

metrics_pd = pd.DataFrame(metrics).drop(columns=['fit_time','score_time'])
mean = metrics_pd.mean()
std  = metrics_pd.std()
cv_summary = pd.concat([mean, std], axis=1, keys=["mean","std"])

cv_summary.sort_values('std', ascending=False)

,mean,std
test_recall,0.543813,0.045711
test_f1,0.592412,0.033298
test_precision,0.652094,0.028635
test_pr_auc,0.661493,0.021661
test_roc_auc,0.846149,0.014045
test_accuracy,0.801918,0.013156


## Cross-Validation Stability Check

A 5-fold stratified cross-validation was run on the training set using the baseline logistic regression pipeline.

The CV results are close to the held-out test results, suggesting that the baseline model performance is reasonably stable and not highly dependent on one particular train/test split.

The CV mean PR-AUC is about 0.66, compared with the no-skill baseline of about 0.27, indicating that the model consistently ranks churn customers above non-churn customers better than random.

# Conclusion
The baseline modeling stage established two reference points.

First, the dummy classifier achieved about 73.5% accuracy by always predicting the majority class, but it failed to identify any churn customers. This confirms that accuracy alone is misleading for this imbalanced classification problem.

Second, the logistic regression baseline substantially improved performance. It achieved about 80.6% accuracy, 55.9% churn recall, 65.7% churn precision, 60.4% churn F1-score, 0.84 ROC-AUC, and 0.63 PR-AUC on the held-out test set.

A 5-fold stratified cross-validation stability check on the training set produced similar results, suggesting that the logistic regression baseline is reasonably stable.

The main limitation of the baseline is that, using the default threshold of 0.5, it still misses a substantial fraction of churn customers. Future work should therefore focus on cross-validation-based tuning, feature engineering, and threshold analysis rather than relying only on default classification behavior.